# 07 - Perform DE analysis per condition

In [ ]:
%autosave 0

In [ ]:
import os
import sys
import dill
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")
sys.path.append(str(base_proj_dir))

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from adjustText import adjust_text
from dynaconf import Dynaconf
from tqdm import tqdm

In [ ]:
# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)
tables_dir = Path(settings.ANALYSIS.tables_dir)

output_anndata_dir = anndata_dir / "tf_ko_panel_de_results_per_annotation_less_stringent"
output_anndata_dir.mkdir(parents=True, exist_ok=True)

output_tables_dir = tables_dir / "annotation_de_results_less_stringent"
output_tables_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Setup keys
sample_key = 'sample'
groupby_key = 'annotation'
condition_key = 'condition'

## Load data

In [ ]:
# Load entire dataset (clustering based annotations)
adata = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated.h5ad")

In [ ]:
adata.obs["annotation"] = (
    adata.obs["annotation"]
    .replace({
        "Early EEC Progenitors": "EEC Progenitors",
        "Late EEC Progenitors": "EEC Progenitors"
    })
)

In [ ]:
adata.obs[groupby_key] = adata.obs[groupby_key].astype(str)
adata.obs[sample_key] = adata.obs[sample_key].astype(str)
adata.obs[condition_key] = adata.obs[condition_key].astype(str)

In [ ]:
adata = adata.raw.to_adata()

## Filter genes

#### Filter lowly expressed genes

In [ ]:
sc.pp.filter_genes(adata, min_counts=100)

#### Filter mitochondrial / ribosomal genes, Keep protein coding genes

In [ ]:
from pybiomart import Dataset

# Get all genes
all_genes = list(adata.var_names)

# Remove long intergenic noncoding RNAs and ENSG genes
keep_genes = [g for g in all_genes if not g.startswith("LINC") and not g.startswith("ENSG")]

# Remove mitochondrial genes
mt_genes = [g for g in all_genes if g.startswith("MT-")]
keep_genes = [g for g in keep_genes if g not in mt_genes]

# Remove ribosomal genes
ribo_genes = [g for g in all_genes if g.startswith("RPS") or g.startswith("RPL")]
keep_genes = [g for g in keep_genes if g not in ribo_genes]

# Connect to Ensembl BioMart
dataset = Dataset(name='hsapiens_gene_ensembl', host='http://www.ensembl.org')

# Query all genes with attributes we want
genes = dataset.query(
    attributes=[
        'ensembl_gene_id',
        'external_gene_name',
        'gene_biotype'
    ]
)

# Filter for protein coding
protein_coding = genes[genes['Gene type'] == 'protein_coding']

# Trim to only protein coding genes
keep_genes = [g for g in keep_genes if g in protein_coding["Gene name"].unique()]

In [ ]:
adata = adata[:, keep_genes]

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_filtered.h5ad")

In [ ]:
# save genes that were kept to a text file
with open(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_filtered_genes.txt", "w") as f:
    for gene in adata.var_names:
        f.write(f"{gene}\n")

In [ ]:
# Save all conditions that are not "Control"
conditions = adata.obs[condition_key].unique().tolist()
conditions = [c for c in conditions if c != "Control"]
with open(output_tables_dir / "tf_ko_panel_conditions.txt", "w") as f:
    for condition in conditions:
        f.write(f"{condition}\n")

## Test DE changes induced by each TF KO across all conditions

In [ ]:
import delnx as dx
import submitit
from pathlib import Path

from pydeseq2.dds import DefaultInference, DeseqDataSet
from pydeseq2.ds import DeseqStats

def run_tf_perturbation(tf_perturb, min_cells = 20):
    print(f"Processing TF perturbation: {tf_perturb}")
    
    adata = sc.read(anndata_dir / "tf_ko_panel_contrastiveVI_raw_annotated_filtered.h5ad")
    
    # Subset to control and current perturbation
    adata_ctrl = adata[adata.obs[condition_key] == "Control"].copy()
    adata_pert = adata[adata.obs[condition_key].isin([tf_perturb])].copy()
    
    # Remove all groups in each condition that have less than 20 cells support
    group_counts_ctrl = adata_ctrl.obs[groupby_key].value_counts()
    group_counts_pert = adata_pert.obs[groupby_key].value_counts()

    valid_group_ctrl = group_counts_ctrl[group_counts_ctrl >= min_cells].index
    valid_group_pert = group_counts_pert[group_counts_pert >= min_cells].index

    adata_ctrl = adata_ctrl[adata_ctrl.obs[groupby_key].isin(valid_group_ctrl)].copy()
    adata_pert = adata_pert[adata_pert.obs[groupby_key].isin(valid_group_pert)].copy()

    # Now for the same groups we test for gene expression 
    common_groups = set(adata_ctrl.obs[groupby_key]).intersection(set(adata_pert.obs[groupby_key]))

    # Put back into one object for both ctrl and pert
    joined_adata = adata_ctrl.concatenate(adata_pert, batch_key=condition_key, batch_categories=['Control', tf_perturb])
    # Subset to only common groups
    joined_adata = joined_adata[joined_adata.obs[groupby_key].isin(common_groups)].copy()
    print(f"Number of common groups between control and {tf_perturb}: {len(common_groups)}")
    print(f"Number of cells in joined dataset: {joined_adata.n_obs}")
    print(f"Number of genes in joined dataset: {joined_adata.n_vars}")
    joined_adata.write(output_anndata_dir / f"tf_ko_panel_{tf_perturb}_vs_control_joined_adata.h5ad")
    
    # Test for all groups
    groups_de_results_sc = []
    groups_de_results_pb = []
    for group in common_groups:
        print(f"Processing milestone: {group}")
        cells_ctrl = adata_ctrl[adata_ctrl.obs[groupby_key] == group]
        cells_pert = adata_pert[adata_pert.obs[groupby_key] == group]

        # Remove genes that are too lowly expressed in both condition
        ctrl_fltr_genes_bool, _ = sc.pp.filter_genes(cells_ctrl, min_cells=3, inplace=False)
        pert_fltr_genes_bool, _ = sc.pp.filter_genes(cells_pert, min_cells=3, inplace=False)
        
        # Find common genes
        common_genes = set(cells_ctrl.var_names[ctrl_fltr_genes_bool]).union(set(cells_pert.var_names[pert_fltr_genes_bool]))
        print(f"Number of common genes after filtering: {len(common_genes)}")
        
        cells_ctrl = cells_ctrl[:, list(common_genes)]
        cells_pert = cells_pert[:, list(common_genes)]

        # Now join the two datasets for comparison
        combined_adata = cells_ctrl.concatenate(cells_pert, batch_key=condition_key, batch_categories=['Control', tf_perturb])
        combined_adata.obs[condition_key] = combined_adata.obs[condition_key].astype("category").cat.reorder_categories(["Control", tf_perturb])
        
        combined_adata = combined_adata[list(combined_adata.obs.sort_values(condition_key).index)]
        print(f"Number of cells in combined dataset: {combined_adata.n_obs}")
        print(f"Number of genes in combined dataset: {combined_adata.n_vars}")
        
        # Escape group name for file saving
        group_safe = group.replace(" ", "_").replace("/", "_").replace("\\", "_")
        combined_adata.write(output_anndata_dir / f"tf_ko_panel_{tf_perturb}_vs_control_group_{group_safe}_adata.h5ad")
        
        # Place counts in a layer for DE testing
        combined_adata.layers["counts"] = combined_adata.X.copy()
    
        # Run differential expression analysis between conditions using negbinom test
        de_results = dx.tl.de(
            combined_adata,
            method="negbinom",
            layer="counts",
            reference=("Control", tf_perturb),
            mode="1_vs_1",
            condition_key=condition_key, 
        )
        de_results[groupby_key] = group
        # Some genes have very low expression leading to very high log2FC values, we remove these
        de_results = de_results.query("abs(log2fc) < 10")
        groups_de_results_sc.append(de_results)
        
        # Run differential expression analysis between conditions using pydeseq2
        adata_pb = dx.pp.pseudobulk(
            combined_adata,
            sample_key=sample_key,  
            group_key=groupby_key,  
            layer="counts",
            min_cells=10,  
            n_pseudoreps=2,
            mode="sum",
        )
        
        inference = DefaultInference()
        dds_icpt = DeseqDataSet(
            adata=adata_pb,
            design="~ 0 + condition",
            inference=inference,
        )
        dds_icpt.deseq2()
        
        # Perform statistical test
        contrast = ["condition",  tf_perturb, "Control"]
        stats = DeseqStats(dds_icpt, contrast=contrast, inference=inference)
        stats.summary()
        
        de_results = stats.results_df.reset_index()
        de_results[groupby_key] = group
        groups_de_results_pb.append(de_results)
        
    # Combine all sc results
    de_results = pd.concat(groups_de_results_sc)
    # Write the results
    de_results.to_csv(output_tables_dir / f"de_results_{tf_perturb}_vs_control_per_annotation.tsv", sep="\t", index=False)
    # Combine all pb results
    de_results = pd.concat(groups_de_results_pb)
    # Write the results
    de_results.to_csv(output_tables_dir / f"de_results_{tf_perturb}_vs_control_per_annotation_pb.tsv", sep="\t", index=False)
    
    return f"Finished TF perturbation: {tf_perturb}"

In [ ]:
from pathlib import Path
import submitit

log_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/submitit_logs")
log_dir.mkdir(exist_ok=True)

# Create executor
executor = submitit.AutoExecutor(folder=log_dir)

executor.update_parameters(
    timeout_min=3 * 60,         
    slurm_partition="batch_gpu",
    gres="gpu:1",
    qos="3h",
    cpus_per_task=1,
    mem_gb=100,
)

# Load perturbations
with open("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/tables/annotation_de_results/tf_ko_panel_conditions.txt", "r") as f:
    tf_perturbations = [line.strip() for line in f.readlines()]

# ---- Submit one independent job per TF ----
jobs = {}
for tf in tf_perturbations:
    job = executor.submit(run_tf_perturbation, tf)
    jobs[tf] = job
    print(f"Submitted job for {tf}: {job.job_id}")

## Pool DE results across annotations and conditions

In [ ]:
# Load conditions
with open('/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/tf_ko_screen/panel/tables/annotation_de_results/tf_ko_panel_conditions.txt', "r") as f:
    tf_perturbations = [line.strip() for line in f.readlines()]

In [ ]:
de_results_joined = []
for tf_perturb in tf_perturbations:
    try:
        # Get DE genes
        de_results = pd.read_csv(output_tables_dir / f"de_results_{tf_perturb}_vs_control_per_annotation.tsv", sep="\t")
        # Append condition info
        de_results["condition"] = tf_perturb
        de_results_joined.append(de_results)
    except Exception as e:
        print(f"Could not load DE results for {tf_perturb}: {e}")
# Combine all results
de_results_all = pd.concat(de_results_joined)
# Write the combined results
de_results_all.to_csv(output_tables_dir / f"tf_ko_panel_contrastiveVI_de_results_annotations.tsv", sep="\t", index=False)

In [ ]:
de_results_joined = []
for tf_perturb in tf_perturbations:
    try:
        # Get DE genes
        de_results = pd.read_csv(output_tables_dir / f"de_results_{tf_perturb}_vs_control_per_annotation_pb.tsv", sep="\t")
        # Append condition info
        de_results["condition"] = tf_perturb
        de_results_joined.append(de_results)
    except Exception as e:
        print(f"Could not load DE results for {tf_perturb}: {e}")
# Combine all results
de_results_all = pd.concat(de_results_joined)
# Write the combined results
de_results_all.to_csv(output_tables_dir / f"tf_ko_panel_contrastiveVI_de_results_annotations_pb.tsv", sep="\t", index=False)

In [ ]:
output_tables_dir / f"tf_ko_panel_contrastiveVI_de_results_annotations_pb.tsv"

## Assess DE

In [ ]:
desired_order = [
    "Stem Cells", "TA Cells", "Cycling TA Cells", "Enterocytes",
    "Goblet Cells", "Paneth Cells", "EEC Progenitors", "EC Cells",
    "X Cells", "D Cells", "I/N Cells", "K Cells"
]

condition_order = ["Control", "Rfx3"]

adata = sc.read(output_anndata_dir / f"tf_ko_panel_{condition_order[1]}_vs_control_joined_adata.h5ad")

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["lognorm"] = adata.X.copy()

# keep only desired cell types
actual_celltypes = [
    ct for ct in desired_order
    if ct in adata.obs[groupby_key].cat.categories
]

adata = adata[adata.obs[groupby_key].isin(actual_celltypes)].copy()

# reorder cell type categories
adata.obs[groupby_key] = adata.obs[groupby_key].cat.reorder_categories(
    actual_celltypes,
    ordered=True
)

# reorder condition categories
adata.obs[condition_key] = pd.Categorical(
    adata.obs[condition_key],
    categories=condition_order,
    ordered=True
)

# create combined label
adata.obs["celltype_condition"] = (
    adata.obs[groupby_key].astype(str)
    + "|"
    + adata.obs[condition_key].astype(str)
)

# propagate desired order to combined category
celltype_condition_order = [
    f"{ct}|{cond}"
    for ct in actual_celltypes
    for cond in condition_order
]

# keep only combinations that actually exist
actual_celltype_condition_order = [
    x for x in celltype_condition_order
    if x in adata.obs["celltype_condition"].values
]

adata.obs["celltype_condition"] = pd.Categorical(
    adata.obs["celltype_condition"],
    categories=actual_celltype_condition_order,
    ordered=True
)

In [ ]:
# Get number of cells for each cell type and condition combination
celltype_condition_counts = adata.obs["celltype_condition"].value_counts().sort_index()
print("Number of cells for each cell type and condition combination:")
print(celltype_condition_counts)

In [ ]:
sc.pl.violin(adata, keys=["LMX1A"], groupby="celltype_condition", layer="lognorm", jitter=0.4, rotation=90)

## Load correlation results

In [ ]:
corr_df = pd.read_csv(Path(settings.ANALYSIS.tables_dir) / "tf_ko_panel_contrastiveVI_de_results_annotations_effect_size_correlation_matrix.tsv", sep=",", index_col=0)

In [ ]:
top20_up = corr_df.loc["cond_Lmx1b"].sort_values(ascending=False).head(200)
top20_down = corr_df.loc["cond_Lmx1b"].sort_values(ascending=True).head(200)

In [ ]:
sc.pl.matrixplot(adata, var_names=top20_up.index.tolist(), groupby="group", use_raw=False, layer="lognorm", cmap="RdPu", figsize=(10,3), standard_scale="var", save="_cond_Lmx1b_up.pdf")

In [ ]:
sc.pl.matrixplot(adata, var_names=top20_down.index.tolist(), groupby="group", use_raw=False, layer="lognorm", cmap="RdPu", figsize=(10,3), standard_scale="var", save="_cond_Lmx1b_down.pdf")

In [ ]:
top_up = corr_df.loc["int_Lmx1b|EEC Progenitors"].sort_values(ascending=False).head(50)
top_down = corr_df.loc["int_Lmx1b|EEC Progenitors"].sort_values(ascending=True).head(50)

In [ ]:
sc.pl.matrixplot(adata, var_names=top_up.index.tolist(), groupby="group", use_raw=False, layer="lognorm", cmap="RdPu", figsize=(10,3), standard_scale="var", save="_int_Lmx1b_EEC_Progenitors_up.pdf")

In [ ]:
sc.pl.matrixplot(adata, var_names=top_down.index.tolist(), groupby="group", use_raw=False, layer="lognorm", cmap="RdPu", figsize=(10,3), standard_scale="var", save="_int_Lmx1b_EEC_Progenitors_down.pdf")

## Showcase selected TFs that are DE upon knockout

In [ ]:
tfs = pd.read_csv("/projects/site/pred/ihb-g-deco/PUBLIC_DATA/DB/TF_names_v_1.01.txt", sep="\t", index_col=0, header=None).index.tolist()
tfs = [tf for tf in tfs if tf in corr_df.columns]

In [ ]:
top_up = corr_df.loc["cond_Lmx1b"].loc[tfs].sort_values(ascending=False).head(50)
top_down = corr_df.loc["cond_Lmx1b"].loc[tfs].sort_values(ascending=True).head(50)

In [ ]:
sc.pl.matrixplot(adata, var_names=top_up.index.tolist() + top_down.index.tolist(), groupby="group", use_raw=False, layer="lognorm", cmap="RdPu", figsize=(5,3), standard_scale="var", save="_int_Lmx1b_EEC_Progenitors_topTFs.pdf")

In [ ]:
top_up = corr_df.loc["int_Lmx1b|EEC Progenitors"].loc[tfs].sort_values(ascending=False).head(10)
top_down = corr_df.loc["int_Lmx1b|EEC Progenitors"].loc[tfs].sort_values(ascending=True).head(10)

In [ ]:
sc.pl.matrixplot(adata, var_names=top_up.index.tolist() + top_down.index.tolist(), groupby="group", use_raw=False, layer="lognorm", cmap="RdPu", figsize=(5,3), standard_scale="var", save="_int_Lmx1b_EEC_Progenitors_topTFs.pdf")